# Chapter 9: Retrieval — Putting It to Work
## Topic 5: RAG Chain Patterns — Stuff, Map-Reduce, Refine, Map-Rerank

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Chapter 8 Topic 1 (Prompt Construction) established the token-budgeting discipline for fitting retrieved chunks into a single prompt — but it implicitly assumed all retrieved chunks get combined into one generation call. This topic names and formalizes the actual *pattern* used for that combination, and introduces three genuine alternatives for cases where "everything in one prompt" breaks down.
- These four patterns (Stuff, Map-Reduce, Refine, Map-Rerank) are a standard taxonomy for how a RAG system turns *multiple* retrieved chunks into *one* final answer — they differ in whether the model sees all chunks at once or one at a time, and in how many model calls the whole process costs.
- **Stuff:** put all retrieved chunks directly into one prompt, one generation call. This is what every earlier chapter's examples have implicitly done — "stuffing" the context block full of chunks and generating once.
- **Map-Reduce:** run a separate generation call on *each* chunk individually (the "map" step, producing one partial answer per chunk), then run one more generation call that combines all those partial answers into a final answer (the "reduce" step).
- **Refine:** process chunks sequentially — generate an initial answer from the first chunk, then feed that answer plus the next chunk back to the model and ask it to *refine* the answer, repeating this one chunk at a time until every chunk has been incorporated.
- **Map-Rerank:** run a separate generation call on each chunk individually, asking each call to both answer the question *and* output a confidence score for how well the chunk actually answered it, then pick the single highest-confidence answer as the final result — no combination step at all, just selection.


### 2. Internal Working — Step by Step

**Stuff — the default, single-call pattern:**
1. Retrieve top-k chunks (Chapter 7's pipeline).
2. Construct one prompt containing all chunks (Chapter 8 Topic 1's token budgeting).
3. One generation call produces the final answer.
- Cost: exactly one model call. Limitation: bounded entirely by the context window — once retrieved content exceeds the available token budget, chunks must be dropped (exactly Chapter 8 Topic 1's greedy-drop behavior), and the model must synthesize across everything it *did* see in a single pass, which gets harder as the number of included chunks grows.

**Map-Reduce — parallelizable, scales past context-window limits:**
1. Retrieve chunks.
2. **Map step:** for each chunk independently, run a generation call: "given only this chunk, answer the question (or say it's not addressed here)." This produces one partial answer per chunk.
3. **Reduce step:** run one more generation call that takes all partial answers and combines them into a single final answer.
- Cost: N+1 model calls for N chunks. The map step is embarrassingly parallel (every chunk's call is independent of every other), so wall-clock latency doesn't have to scale linearly with N if calls run concurrently — though total cost does. Limitation: a fact that only makes sense in combination with another chunk (e.g., one chunk states a rate, another states a condition that modifies it) can get lost, since each map call only sees one chunk in isolation, with no way to reason across chunks until the reduce step, which only has each chunk's already-summarized partial answer to work with, not the original text.

**Refine — sequential, chunk-by-chunk accumulation:**
1. Retrieve chunks, establish an order (typically the ranked retrieval order).
2. Generate an initial answer from the first chunk alone.
3. For each subsequent chunk, feed the model the current answer plus the new chunk, and ask it to refine (update, correct, or extend) the answer given this new information.
4. After the last chunk, the final refined answer is returned.
- Cost: N model calls for N chunks, but inherently sequential — each call depends on the previous one's output, so this cannot be parallelized like Map-Reduce's map step. Limitation: the answer's quality can become sensitive to chunk order (an earlier topic in this chapter — Chapter 7's discussion of "lost in the middle" — is relevant here too, since later refinement calls may drift away from or overwrite earlier correct information without careful prompting).

**Map-Rerank — parallel generation, selection instead of combination:**
1. Retrieve chunks.
2. For each chunk independently, run a generation call that produces both an answer attempt *and* a self-reported confidence score for how well this specific chunk actually answers the question.
3. Select the single answer with the highest confidence score as the final result — no combination step.
- Cost: N model calls, parallelizable like Map-Reduce's map step. Limitation: works well specifically when the correct answer is expected to come from a *single* chunk (a lookup-style question), and works poorly for questions that genuinely need synthesis across multiple chunks, since it only ever returns one chunk's answer, never a combination.


### 3. How This Is Implemented in This Project

- **Stuff** is this project's default and correct choice for the vast majority of queries: this project's actual retrieved-chunk counts (Chapter 7's top-k after reranking and MMR) are small — a handful of short policy chunks — comfortably fitting within a single prompt's token budget (Chapter 8 Topic 1), with no need for the added complexity of the other three patterns.
- **Map-Reduce or Refine** become relevant only if a future use case required synthesizing across many more chunks than currently retrieved — for example, if a customer's question required consulting every SOP step across a much longer procedural document than the current corpus contains, or if the knowledge base grew substantially larger and a single query's genuinely relevant content routinely exceeded what Stuff's context budget could hold.
- **Map-Rerank** connects directly to this project's existing FD-reference-lookup pattern (Chapter 3's `validate_fd_reference`, Topic 4's exact-lookup discussion) — for a genuinely single-fact lookup question ("what is the exact penalty percentage"), a Map-Rerank-style pattern (find the one chunk that most directly and confidently answers this, use only that one) may actually produce a cleaner, more precisely-cited answer than Stuff's approach of handing the model several chunks and asking it to synthesize, when synthesis isn't actually needed for a single-fact question.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Cost scaling is the central practical trade-off between these patterns.** Stuff costs one call regardless of chunk count (until the context window itself becomes the limit). Map-Reduce, Refine, and Map-Rerank all cost roughly N calls for N chunks — at meaningful chunk counts, this is a direct, multiplicative cost increase over Stuff, and should only be adopted when Stuff's single-call limitation genuinely can't be worked around by simply retrieving fewer, better-ranked chunks (Chapter 7's reranking and MMR already exist specifically to make a small top-k as good as possible, reducing the pressure to need more than Stuff can handle).
- **Latency implications differ meaningfully even among the multi-call patterns.** Map-Reduce and Map-Rerank's map/generation steps are independent per chunk and can run concurrently, bounding wall-clock latency close to a single call's latency if the infrastructure supports true parallel requests. Refine is inherently sequential — each step depends on the previous step's output — so its wall-clock latency scales directly with chunk count, with no possibility of parallelization, making it the least attractive choice on pure latency grounds among the multi-call options.
- **Refine's order-sensitivity is a genuine, measurable failure mode:** since later refinement steps operate on an already-generated answer rather than fresh raw context, a later chunk containing a correction to something stated in an earlier chunk depends entirely on the model successfully incorporating that correction during refinement — this should be validated with retrieval-evaluation-style testing (Chapter 7 Topic 9's discipline) on ordering-sensitive test cases specifically, not assumed to work correctly.
- **Map-Reduce's reduce step can be a hidden faithfulness risk:** the reduce step's generation call operates on partial *summaries* rather than the original source text — this is one level further removed from the original grounding than Stuff's single call, meaning Chapter 8's faithfulness verification (checking claims against retrieved context) may need to check against the intermediate partial answers as well as the original chunks to catch a faithfulness failure introduced specifically during reduction.
- **Monitoring:** if more than one pattern is used in production (Stuff by default, escalating to another pattern under specific conditions), track chunk-count distribution against whichever pattern-selection threshold is used, and track cost and latency broken down by pattern — this reveals whether the escalation trigger is well-calibrated, or firing more or less often than the underlying data actually requires.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Stuff vs escalating to a multi-call pattern:** Stuff should always be the default, given its single-call cost efficiency, and escalation to a more expensive pattern should be evidence-based — triggered specifically when retrieved content genuinely exceeds the available token budget (Chapter 8 Topic 1), not adopted preemptively "in case" a future query needs it.
- **Map-Reduce vs Refine vs Map-Rerank, once escalation past Stuff is genuinely needed:** Map-Reduce is the right choice when the question needs synthesis across chunks and parallel execution is available to control latency. Refine is the right choice specifically when chunk order carries real informational significance (like a sequential SOP where later steps modify or depend on earlier ones) and the sequential dependency this creates is actually appropriate to the content's structure, not just an accepted cost. Map-Rerank is the right choice when the question is fundamentally a single-fact lookup and only one chunk is expected to contain the real answer, making a combination step actively unnecessary or even harmful (risking diluting a precise answer with irrelevant synthesis from unrelated chunks).
- **The real dilemma when a use case's chunk-count needs genuinely exceed what Stuff can handle:** this is often a signal to revisit earlier pipeline stages (has retrieval or reranking, Chapter 7, been tuned well enough to make a smaller top-k sufficient? has chunking strategy, Chapter 5/9 Topic 10, been optimized to reduce redundant content reaching this stage at all?) before reaching for a more expensive generation-side pattern — the more expensive chain patterns are a genuine, valid tool, but should be a considered choice after the earlier, cheaper levers have been examined, not the first reach when a context-budget problem appears.


### 6. Alternatives and When to Use Each

- **Stuff:** the default choice for the vast majority of queries where retrieved content comfortably fits the available token budget — this project's actual, current situation given a small top-k and short policy chunks.
- **Map-Reduce:** appropriate when synthesis across genuinely many chunks is needed and parallel execution is available to bound latency — worth adopting only once a specific use case demonstrably needs more chunks than Stuff can hold.
- **Refine:** appropriate specifically when chunk order carries genuine informational dependency (later content modifies or builds on earlier content) — accept its sequential latency cost only when this ordering property is actually present and meaningful.
- **Map-Rerank:** appropriate for single-fact lookup questions where exactly one chunk is expected to contain the real answer, and combining multiple chunks would add noise rather than value.


### 7. Common Mistakes and Production Failures

- Adopting a multi-call pattern (Map-Reduce, Refine, or Map-Rerank) preemptively, before actually confirming that Stuff's single-call context budget is genuinely insufficient for real production queries — this multiplies cost with no corresponding benefit if Stuff was already handling the actual workload fine.
- Using Refine for content with no genuine sequential dependency, paying its sequential-latency cost for no benefit over the parallelizable Map-Reduce pattern.
- Using Map-Rerank for questions that genuinely need synthesis across multiple chunks, producing an answer based on only one chunk's perspective when the complete, correct answer required combining several.
- Not extending faithfulness verification (Chapter 8 Topic 4) to check Map-Reduce's intermediate partial answers, missing a faithfulness failure introduced during the reduce step rather than the original map step.
- Reaching for a more expensive multi-call generation pattern before checking whether better retrieval, reranking, or chunking (the earlier, cheaper pipeline stages) could have made Stuff sufficient in the first place.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What's the fundamental difference between Stuff and the other three chain patterns?
  A: Stuff combines all retrieved chunks into a single prompt and makes one generation call. The other three (Map-Reduce, Refine, Map-Rerank) each make roughly one generation call per chunk, differing in how those per-chunk results get combined — reduced together, sequentially refined, or selected from by confidence — trading a single-call cost efficiency for the ability to handle more content than a single context window could hold.

- Q: Why is Refine the only one of the four patterns that can't be parallelized?
  A: Each refinement step takes the previous step's output (the current answer) as its input, along with the next chunk — this creates a genuine sequential dependency, where step N cannot start until step N-1 has completed. Map-Reduce's map step and Map-Rerank's per-chunk calls, by contrast, are each independent of one another and only depend on the original chunk and question, so they can run concurrently.

**Intermediate**

- Q: When would Map-Rerank produce a worse answer than Stuff, even for the same retrieved chunks?
  A: When the correct, complete answer genuinely requires synthesizing information from more than one chunk — Map-Rerank only ever returns one chunk's answer (whichever scored highest confidence), with no combination step, so it structurally cannot represent an answer that depends on combining multiple chunks' information the way Stuff (or Map-Reduce) can.

- Q: Why might Map-Reduce's reduce step introduce a faithfulness risk that wasn't present in the original map step?
  A: The reduce step generates its combined answer based on the map step's partial *summaries*, not the original source chunk text — this is one additional level removed from the original grounding. A faithfulness check that only verifies the final answer against the original chunks might miss a subtler failure where the reduce step correctly summarized what the partial answers said, but one of those partial answers had already drifted from the original chunk during the map step.

**Advanced**

- Q: A production system currently uses Stuff, and a new use case requires answering questions that need information from significantly more chunks than the current context budget can hold. Walk through your decision process before choosing a replacement pattern.
  A: Before reaching for a more expensive generation-side pattern, first check whether the earlier pipeline stages can be improved to make Stuff sufficient — is reranking (Chapter 7) tuned well enough to surface a smaller, higher-precision top-k? Would improved chunking strategy reduce redundant content reaching this stage? Only if these cheaper levers are already well-tuned and the use case still genuinely needs more chunks than Stuff's budget allows should a multi-call pattern be adopted — and the specific choice among Map-Reduce, Refine, and Map-Rerank should follow from the content's actual structure (does synthesis need to happen, is ordering meaningful, is this fundamentally a single-fact lookup) rather than defaulting to whichever pattern is most familiar.

- Q: How would you validate that Refine's sequential processing is actually incorporating corrections from later chunks, rather than anchoring too strongly on the initial answer?
  A: Build a labeled test set specifically designed to probe this — cases where an early chunk states something that a later chunk explicitly corrects or modifies — and measure whether the final refined answer reflects the correction. This is the same evidence-based validation discipline as Chapter 7 Topic 9's retrieval evaluation, applied specifically to Refine's core risk: that later refinement steps might not reliably override or update information established earlier, especially if the prompt's refinement instructions aren't carefully designed to explicitly invite correction rather than just extension.

**Scenario-based**

- Q: A teammate proposes switching from Stuff to Map-Reduce across the board, arguing it will make the system "more scalable." How do you respond?
  A: Scalability in what dimension matters here — Map-Reduce trades a single-call cost for N-call cost, which is a real, ongoing cost multiplier at production volume, not a free scalability win. If the actual current workload's chunk counts comfortably fit within Stuff's context budget (which, for this project's small top-k and short policy chunks, they do), switching to Map-Reduce would increase cost and complexity with no corresponding benefit. The right question isn't which pattern sounds more scalable in the abstract, but whether Stuff's specific limitation (context window) is actually being hit by real production queries — if not, Stuff remains the correct, most cost-efficient choice.

**Follow-up questions to expect:**

- "How would you design an automatic pattern-selection mechanism that escalates from Stuff only when genuinely needed?"
- "What monitoring would tell you if Refine's answer quality was degrading due to order-sensitivity?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **These four patterns are a specific instance of a much more general "how do you combine many pieces of information into one output" problem:** map-reduce as a general computational pattern (distinct from this specific RAG application of it) originates from large-scale distributed data processing, and recognizing the RAG chain-pattern version as an application of that older, more general idea helps generalize the concept beyond just this one context.
- **The trade-off between single-call efficiency and multi-call flexibility echoes patterns seen earlier in this notebook series:** BM25's k1 saturation, RRF's rank-based fusion, and reranking's bi-encoder-vs-cross-encoder trade-off all involve some version of "cheap and simple" versus "more expensive but more capable" — recognizing this recurring shape across many different specific techniques is a mark of genuinely internalized, transferable understanding rather than memorized, disconnected facts.
- **Pattern selection itself could be automated as a triggering decision**, directly analogous to Topic 1's retrieval-triggering discussion — a system could measure whether retrieved content exceeds Stuff's available budget and automatically escalate to Map-Reduce or another pattern only when needed, applying the exact same "cheap default, escalate only when a measured condition requires it" philosophy established throughout this chapter.

### 10. Quick Revision Summary (for last-minute interview prep)

> Stuff, Map-Reduce, Refine, and Map-Rerank are the four standard patterns for combining multiple retrieved chunks into one final generated answer. Stuff puts everything in one prompt for one generation call — the cheapest option, and the right default whenever retrieved content fits the available context budget, which is this project's actual, current situation. Map-Reduce processes each chunk independently (parallelizable) then combines the partial answers in a final reduce call — N+1 calls, appropriate when genuine cross-chunk synthesis is needed and content exceeds what Stuff can hold. Refine processes chunks sequentially, updating one running answer — inherently non-parallelizable and order-sensitive, appropriate specifically when chunk order carries real informational dependency. Map-Rerank generates independently per chunk and selects the single highest-confidence result — parallelizable, but structurally incapable of combining information across chunks, so appropriate only for single-fact lookup questions. The multi-call patterns should be adopted only after confirming Stuff's context-budget limitation is genuinely being hit by real queries, and ideally only after checking whether better retrieval, reranking, or chunking could have made Stuff sufficient in the first place — not adopted preemptively based on abstract scalability concerns.


### Module 1: Setup — A Simulated Generation Function and Real Chunk Data

Since we can't call the real Claude API offline, use a simple simulated generator -- but the ORCHESTRATION logic (call counting, sequencing, combination) in every pattern below is real, testable code, not simulated.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup -- simulated generation, REAL orchestration logic
#
# We cannot call a real LLM here. generate() below is a simple
# simulation standing in for client.messages.create(). Everything
# ELSE in this notebook -- how many times each pattern calls generate(),
# in what order, how results get combined -- is REAL, testable
# orchestration logic, which is what this topic is actually about.
# ------------------------------------------------------------------

CALL_COUNT = {"total": 0}

def generate(prompt: str) -> str:
    """SIMULATES a model call. Counts real invocations so we can
    measure each pattern's actual call cost honestly. Checks for
    COMBINED signals first, so a prompt containing BOTH the penalty
    fact AND the exception fact produces a genuinely combined answer --
    this is what makes the synthesis-vs-selection distinction between
    patterns actually observable below, rather than every pattern
    just matching the same first condition regardless of content."""
    CALL_COUNT["total"] += 1
    text = prompt.lower()
    has_penalty = "1 percent" in text and "penalty" in text
    has_exception = "waived" in text or "death" in text
    has_senior = "senior citizen" in text

    if has_penalty and has_exception:
        return ("The penalty is 1 percent on the applicable interest rate, but it is "
                "waived in case of the depositor's death. [confidence: high]")
    elif has_penalty:
        return "The penalty is 1 percent on the applicable interest rate. [confidence: high]"
    elif has_exception:
        return "The penalty is waived in case of the depositor's death. [confidence: high]"
    elif has_senior:
        return "Senior citizens get an additional 0.5 percent interest. [confidence: high]"
    else:
        return "This chunk does not address the question. [confidence: low]"


# chunks retrieved for a genuinely multi-fact question: "what is the
# penalty, and are there any exceptions to it?"
RETRIEVED_CHUNKS = [
    {"text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.", "source": "01_FD_FAQ.pdf"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.", "source": "02_FD_Product_Guide.pdf"},
    {"text": "The premature withdrawal penalty is waived if the FD is closed due to the death of the depositor.", "source": "04_FD_Policy_Reference.pdf"},
]

QUESTION = "What is the premature withdrawal penalty, and are there any exceptions to it?"

print("=" * 70)
print("MODULE 1: SETUP")
print("=" * 70)
print(f"Question: '{QUESTION}'")
print(f"Retrieved chunks: {len(RETRIEVED_CHUNKS)}")
for c in RETRIEVED_CHUNKS:
    chunk_source = c["source"]
    chunk_text = c["text"][:60]
    print(f"  [{chunk_source}] {chunk_text}...")
print("\nNote: the CORRECT complete answer needs BOTH chunk 1 (the penalty)")
print("AND chunk 3 (the death exception) -- chunk 2 is a distractor about")
print("an unrelated topic (senior citizen rates). This tests whether each")
print("pattern can correctly synthesize across chunks vs missing the")
print("exception, or getting distracted by irrelevant content.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: SETUP
Question: 'What is the premature withdrawal penalty, and are there any exceptions to it?'
Retrieved chunks: 3
  [01_FD_FAQ.pdf] Premature withdrawal of FD incurs a 1 percent penalty on the...
  [02_FD_Product_Guide.pdf] Senior citizens receive an additional 0.5 percent interest o...
  [04_FD_Policy_Reference.pdf] The premature withdrawal penalty is waived if the FD is clos...

Note: the CORRECT complete answer needs BOTH chunk 1 (the penalty)
AND chunk 3 (the death exception) -- chunk 2 is a distractor about
an unrelated topic (senior citizen rates). This tests whether each
pattern can correctly synthesize across chunks vs missing the
exception, or getting distracted by irrelevant content.

Module 1 complete. Run Module 2 next.


### Module 2: Stuff vs Map-Reduce

Implement both patterns for real, run them on the same chunks, and measure actual call counts and whether each correctly captures the exception.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Stuff vs Map-Reduce -- real orchestration, measured cost
# ------------------------------------------------------------------

def run_stuff(chunks: list, question: str) -> dict:
    """STUFF: all chunks in ONE prompt, ONE generation call."""
    combined_context = "\n---\n".join(f"[{c['source']}] {c['text']}" for c in chunks)
    prompt = f"Context:\n{combined_context}\n\nQuestion: {question}"
    calls_before = CALL_COUNT["total"]
    answer = generate(prompt)
    calls_used = CALL_COUNT["total"] - calls_before
    return {"answer": answer, "calls_used": calls_used, "pattern": "Stuff"}


def run_map_reduce(chunks: list, question: str) -> dict:
    """MAP-REDUCE: one call PER chunk (map), then one call to combine
    all partial answers (reduce). REAL call counting, REAL sequencing."""
    calls_before = CALL_COUNT["total"]

    # MAP step -- one independent call per chunk
    partial_answers = []
    for c in chunks:
        map_prompt = f"Context: [{c['source']}] {c['text']}\n\nQuestion: {question}"
        partial = generate(map_prompt)
        partial_answers.append(f"From {c['source']}: {partial}")

    # REDUCE step -- one call combining all partial answers
    combined_partials = "\n".join(partial_answers)
    reduce_prompt = f"Partial answers from different sources:\n{combined_partials}\n\nCombine these into one final answer to: {question}"
    final_answer = generate(reduce_prompt)

    calls_used = CALL_COUNT["total"] - calls_before
    return {"answer": final_answer, "calls_used": calls_used, "pattern": "Map-Reduce",
            "partial_answers": partial_answers}


print("=" * 70)
print("STUFF vs MAP-REDUCE -- real call counts")
print("=" * 70)

stuff_result = run_stuff(RETRIEVED_CHUNKS, QUESTION)
print(f"\n[STUFF]")
print(f"  Calls used: {stuff_result['calls_used']}")
print(f"  Answer: {stuff_result['answer']}")

map_reduce_result = run_map_reduce(RETRIEVED_CHUNKS, QUESTION)
print(f"\n[MAP-REDUCE]")
print(f"  Calls used: {map_reduce_result['calls_used']}")
print(f"  Partial answers per chunk:")
for pa in map_reduce_result["partial_answers"]:
    print(f"    {pa}")
print(f"  Final (reduced) answer: {map_reduce_result['answer']}")

print(f"\nSTUFF used {stuff_result['calls_used']} call(s).")
print(f"MAP-REDUCE used {map_reduce_result['calls_used']} call(s) for the same {len(RETRIEVED_CHUNKS)} chunks --")
print(f"exactly the N+1 cost the theory predicts ({len(RETRIEVED_CHUNKS)} chunks + 1 reduce call).")
print("\nModule 2 complete. Run Module 3 next.")


STUFF vs MAP-REDUCE -- real call counts

[STUFF]
  Calls used: 1
  Answer: The penalty is 1 percent on the applicable interest rate, but it is waived in case of the depositor's death. [confidence: high]

[MAP-REDUCE]
  Calls used: 4
  Partial answers per chunk:
    From 01_FD_FAQ.pdf: The penalty is 1 percent on the applicable interest rate. [confidence: high]
    From 02_FD_Product_Guide.pdf: Senior citizens get an additional 0.5 percent interest. [confidence: high]
    From 04_FD_Policy_Reference.pdf: The penalty is waived in case of the depositor's death. [confidence: high]
  Final (reduced) answer: The penalty is 1 percent on the applicable interest rate, but it is waived in case of the depositor's death. [confidence: high]

STUFF used 1 call(s).
MAP-REDUCE used 4 call(s) for the same 3 chunks --
exactly the N+1 cost the theory predicts (3 chunks + 1 reduce call).

Module 2 complete. Run Module 3 next.


### Module 3: Refine vs Map-Rerank

Implement the remaining two patterns and directly compare all four on cost and on whether they correctly handle the multi-chunk exception case.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Refine vs Map-Rerank -- real sequential vs real selection
# ------------------------------------------------------------------

def run_refine(chunks: list, question: str) -> dict:
    """REFINE: sequential, one chunk at a time, each call depends on
    the PREVIOUS call's output. Genuinely cannot be parallelized --
    this loop structure enforces that dependency for real."""
    calls_before = CALL_COUNT["total"]

    current_answer = None
    for i, c in enumerate(chunks):
        if current_answer is None:
            prompt = f"Context: [{c['source']}] {c['text']}\n\nQuestion: {question}"
        else:
            prompt = (f"Current answer: {current_answer}\n\n"
                      f"New context: [{c['source']}] {c['text']}\n\n"
                      f"Refine the current answer given this new context, for: {question}")
        current_answer = generate(prompt)  # each call LITERALLY depends on the previous result

    calls_used = CALL_COUNT["total"] - calls_before
    return {"answer": current_answer, "calls_used": calls_used, "pattern": "Refine"}


def run_map_rerank(chunks: list, question: str) -> dict:
    """MAP-RERANK: one call per chunk, each producing an answer +
    confidence; SELECT the highest-confidence one. No combination step."""
    calls_before = CALL_COUNT["total"]

    candidate_answers = []
    for c in chunks:
        prompt = f"Context: [{c['source']}] {c['text']}\n\nQuestion: {question}"
        answer = generate(prompt)
        confidence = "high" if "high" in answer else "low"
        candidate_answers.append({"source": c["source"], "answer": answer, "confidence": confidence})

    # SELECT the first "high" confidence answer (real selection logic,
    # standing in for a numeric confidence score comparison)
    high_confidence = [a for a in candidate_answers if a["confidence"] == "high"]
    selected = high_confidence[0] if high_confidence else candidate_answers[0]

    calls_used = CALL_COUNT["total"] - calls_before
    return {"answer": selected["answer"], "calls_used": calls_used, "pattern": "Map-Rerank",
            "selected_source": selected["source"], "all_candidates": candidate_answers}


refine_result = run_refine(RETRIEVED_CHUNKS, QUESTION)
map_rerank_result = run_map_rerank(RETRIEVED_CHUNKS, QUESTION)

print("=" * 70)
print("REFINE vs MAP-RERANK -- real orchestration")
print("=" * 70)

print(f"\n[REFINE] calls used: {refine_result['calls_used']} (sequential, cannot parallelize)")
print(f"  Final answer: {refine_result['answer']}")

print(f"\n[MAP-RERANK] calls used: {map_rerank_result['calls_used']} (parallelizable)")
print(f"  Selected from: {map_rerank_result['selected_source']}")
print(f"  Final answer: {map_rerank_result['answer']}")
print(f"  (Note: only ONE chunk's answer was used -- no combination happened)")

print("\n" + "=" * 70)
print("ALL FOUR PATTERNS -- SIDE BY SIDE")
print("=" * 70)
all_results = [stuff_result, map_reduce_result, refine_result, map_rerank_result]
print(f"{'Pattern':<14} | {'Calls used':>10} | {'Mentions death exception?':>27}")
print("-" * 60)
for r in all_results:
    mentions_exception = "death" in r["answer"].lower() or "waived" in r["answer"].lower()
    print(f"{r['pattern']:<14} | {r['calls_used']:>10} | {str(mentions_exception):>27}")

print("\nNotice MAP-RERANK selected only ONE chunk's answer and therefore")
print("could NOT combine the base penalty fact with the death exception --")
print("exactly the theory's predicted limitation for questions needing")
print("genuine cross-chunk synthesis, demonstrated with real, measured")
print("call counts and real answer content, not just asserted.")

print("\nModule 3 complete. All Chapter 9 Topic 5 modules done.")
print()
print("=" * 70)
print("CHAPTER 9 TOPIC 5 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Stuff: 1 call, all chunks together -- cheapest, right default when
  content fits the context budget.

  Map-Reduce: N+1 calls (map each chunk + 1 reduce) -- parallelizable
  map step, handles synthesis across many chunks past Stuff's limit.

  Refine: N calls, but SEQUENTIAL -- cannot be parallelized, each step
  depends on the previous step's output, order-sensitive.

  Map-Rerank: N calls, parallelizable, but SELECTS one chunk's answer
  rather than combining -- wrong choice when synthesis across chunks
  is genuinely needed (demonstrated concretely in Module 3).

  Always confirm Stuff's context-budget limit is ACTUALLY being hit
  by real queries before adopting a more expensive multi-call pattern.
""")


REFINE vs MAP-RERANK -- real orchestration

[REFINE] calls used: 3 (sequential, cannot parallelize)
  Final answer: The penalty is 1 percent on the applicable interest rate, but it is waived in case of the depositor's death. [confidence: high]

[MAP-RERANK] calls used: 3 (parallelizable)
  Selected from: 01_FD_FAQ.pdf
  Final answer: The penalty is 1 percent on the applicable interest rate. [confidence: high]
  (Note: only ONE chunk's answer was used -- no combination happened)

ALL FOUR PATTERNS -- SIDE BY SIDE
Pattern        | Calls used |   Mentions death exception?
------------------------------------------------------------
Stuff          |          1 |                        True
Map-Reduce     |          4 |                        True
Refine         |          3 |                        True
Map-Rerank     |          3 |                       False

Notice MAP-RERANK selected only ONE chunk's answer and therefore
could NOT combine the base penalty fact with the death exception 